### **Longstaff-Schwartz Monte Carlo**

Idea w skrócie: standardowe Monte Carlo nie jest w stanie wziąć pod uwagę faktu, że dla opcji amerykańskich knock-and-out istnieją optymalne momenty wykonania przed momentem zapadnięcia. Aby to wziąć pod uwagę w symulacji, zakładamy, że wartość kontynuacji opcji w danym momencie jesteśmy w stanie przybliżyć kombinacją liniową pewnych funkcji bazowych $\varphi_1, ... \varphi_n$:
$$
V_{m - 1}^{\text{(kontynuacja)}} = \mathbb{E}[V_m | S_{m - 1}] \approx \sum_{i = 1}^n \beta_i \varphi_i (S_{m - 1})
$$
gdzie współczynniki $\beta_1, ... \beta_n$ są dobrane tak, aby zminimalizować błąd średniokwadratowy między $V_m$, a $\sum_{i = 1}^n \beta_i \varphi_i (S_{m - 1})$

Opis algorytmu (też w skrócie...): Symulujemy notowania  (GBMy) w mierze neutralnej na ryzyko. Liczymy payoff dla końcowych wartości (sprawdzając, czy nie trafiliśmy w barierę). Teraz, wyznaczamy wartości od końca. Sprawdzamy, które ścieżki są *in-the-money* i nie trafiły w barierę. Dla tych ścieżek zapisujemy wartość aktywa bazowego z obecnej chwili jako $S$, a wartość opcji z poprzedniej chwili jako $V$. Tak jak napisalismy wyżej, zakładamy, że tę wartość (czyli wartość kontynuacji, nie wykonania) możemy przybliżyć korzystając z kombinacji liniowej pewnych wybranych funkcji bazowych:
$$
V \approx \sum_{i = 1}^n \beta_i \varphi_i (S) = \beta_1 \varphi_1 (S) + ... + \beta_n \varphi_n (S)
$$
i szukamy (funkcja w *numpy* szuka) współczynników $\beta_1, ... \beta_n$ tak, aby zminimalizować błąd średniokwadratowy i wyznaczamy dla każdej ścieżki wartość kontynuacji według otrzymanych współczynników. Następnie sprawdzamy, czy tak wyestymowana wartość kontynuacji jest większa od wartości natychmiastowego wykonania. Jeśli wartość natychmiastowego wykonania jest większa, to przyjmujemy ją jako wartość opcji; w przeciwnym wypadku, dyskontujemy wartość opcji z poprzedniej chwili i przyjmujemy ją jako wartość opcji. Kontynuujemy tę procedurę aż dojdziemy do chwili początkowej. Na końcu, bierzemy średnią z uzyskanych wartości i otrzymujemy cenę opcji.

Poniższa funkcja wyznacza wartość według tego algorytmu dla opcji call i przy założeniu ciągu trzech funkcji bazowych: $\varphi_1(S) = 1, \varphi_2(S) = S, \varphi_3(S) = S^2$.

In [2]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def longstaff_schwartz_monte_carlo(
    option_strike: float,
    option_barrier: float,
    S0: float,
    volatility: float,
    risk_free_rate: float,
    time_to_maturity: float,
    n_steps: int = 250,
    n_paths: int = 20000,
    seed: int = 42
):
    def simulate_gbm(
        S0: float,
        volatility: float,
        drift: float,
        t: float,
        n: float,
        M: float
    ):
        dt = t / n
        brownian_increments = np.random.normal(loc = 0, scale = np.sqrt(dt), size = n*M).reshape(M, n)
        brownian_increments_transformed = (drift - volatility**2 / 2) * np.repeat(np.arange(1, n + 1), M).reshape(M, n)*dt + volatility * np.cumsum(brownian_increments, 1)
        brownian_increments_transformed = np.column_stack((np.ones(M).reshape(M, 1), np.exp(brownian_increments_transformed)))
        
        return S0*brownian_increments_transformed
    
    np.random.seed(seed)
    
    time_step = time_to_maturity / n_steps
    discount_rate = np.exp(-risk_free_rate * time_step)
    
    S = simulate_gbm(
        S0 = S0,
        volatility = volatility,
        drift = risk_free_rate,
        t = time_to_maturity,
        n = n_steps,
        M = n_paths
    )
    
    safe_paths = np.ones_like(S, dtype=bool)
    for t in range(1, n_steps + 1):
        safe_paths[:, t] = safe_paths[:, t-1] & (S[:, t] < option_barrier)
    
    option_values = np.maximum(S[:, -1] - option_strike, 0)
    option_values[~safe_paths[:, -1]] = 0
    
    for t in range(n_steps - 1, 0, -1):
        
        safe_t = safe_paths[:, t]
        in_the_money_paths = (S[:, t] > option_strike) & safe_t
        
        if np.sum(in_the_money_paths) == 0:
            option_values = discount_rate * option_values
            continue
        
        current_price_values = S[in_the_money_paths, t]
        previous_option_values = discount_rate * option_values[in_the_money_paths]
        
        transformed_current_price_values = np.vstack([np.ones_like(current_price_values), current_price_values, current_price_values**2]).T
        coeffs = np.linalg.lstsq(transformed_current_price_values, previous_option_values, rcond=None)[0]
        
        continuation_estimate = coeffs[0] + coeffs[1]*current_price_values + coeffs[2]*current_price_values**2
        exercise_values = current_price_values - option_strike
        
        option_values[in_the_money_paths] = np.where(exercise_values > continuation_estimate, exercise_values, previous_option_values)
        
        option_values[~safe_t] = 0.0
        
        option_values = discount_rate * option_values
    
    option_price = np.mean(option_values)
    
    return option_price

In [4]:
price = longstaff_schwartz_monte_carlo(
    option_strike=3200,
    option_barrier=3600,
    S0 = 3200,
    volatility=0.2,
    risk_free_rate=0.05,
    time_to_maturity=1,
    n_steps=10000,
    n_paths=50000
)

In [5]:
print(f"Cena opcji: {price:.4f}")

Cena opcji: 235.1055


In [ ]:
ls_alg_vectorized = np.vectorize(pyfunc = longstaff_schwartz_monte_carlo,
                                 excluded={'option_strike', 'option_barrier', 'volatility', 'risk_free_rate', 'time_to_maturity',
                                           'n_steps', 'n_paths', 'seed'})